# 🚀 AI Product Intelligence System
### Gen AI Bootcamp — Day 2 Homework

**Tasks Covered:**
- ✅ Task 1: Smart Product Recommendation Engine
- ✅ Task 2: Unique Product Catalog Creation (Deduplication)
- ✅ Task 3: Reverse Product Search (Text → Products)

**Tech Stack:** CLIP · Sentence-BERT · ChromaDB · scikit-learn · matplotlib · Kaggle Fashion Dataset

## 📦 Install Dependencies

In [1]:
pip install -q transformers sentence-transformers chromadb pillow scikit-learn matplotlib seaborn pandas numpy torch torchvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 📥 Load Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load Kaggle fashion dataset
# Dataset: https://www.kaggle.com/code/sahandakramipour/fashion-product-images-small
styles_path ='/kaggle/input/fashion-product-images-small/styles.csv'

df = pd.read_csv(styles_path)
df = df.dropna(subset=['productDisplayName', 'masterCategory', 'subCategory', 'articleType', 'baseColour'])
df = df.reset_index(drop=True)

# Build display text for each product
df['product_text'] = (
    df['baseColour'].fillna('') + ' ' +
    df['articleType'].fillna('') + ' - ' +
    df['productDisplayName'].fillna('')
).str.strip()

print(f'Loaded {len(df)} products')
print(df[['id', 'productDisplayName', 'masterCategory', 'subCategory', 'articleType', 'baseColour']].head(5))

FileNotFoundError: File not found: /kaggle/input/fashion-product-images-small/styles.csv

## 🔧 Load CLIP & Sentence-BERT Models

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# CLIP — for image+text embeddings (Tasks 1 & 3)
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

# Sentence-BERT — for text-only semantic similarity (Task 2)
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print('✅ Models loaded')

## 🧮 Compute Embeddings

In [ ]:
from sklearn.preprocessing import normalize

# Use a sample for speed; remove slice for full dataset
SAMPLE_SIZE = 500
df_sample = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# --- CLIP Text Embeddings ---
def get_clip_text_embeddings(texts, batch_size=64):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = clip_processor(text=batch, return_tensors='pt', padding=True, truncation=True, max_length=77).to(device)
        with torch.no_grad():
            embs = clip_model.get_text_features(**inputs)
        all_embs.append(embs.cpu().numpy())
    return normalize(np.vstack(all_embs))

# --- Sentence-BERT Embeddings ---
def get_sbert_embeddings(texts):
    embs = sbert_model.encode(texts, show_progress_bar=True, batch_size=64)
    return normalize(embs)

print('Computing CLIP text embeddings...')
clip_embeddings = get_clip_text_embeddings(df_sample['product_text'].tolist())

print('Computing SBERT embeddings...')
sbert_embeddings = get_sbert_embeddings(df_sample['product_text'].tolist())

print(f'✅ CLIP: {clip_embeddings.shape} | SBERT: {sbert_embeddings.shape}')

## 🗄️ Build ChromaDB Vector Store

In [ ]:
import chromadb

client = chromadb.Client()

# CLIP collection
try: client.delete_collection('products_clip')
except: pass
clip_collection = client.create_collection('products_clip', metadata={'hnsw:space': 'cosine'})

# SBERT collection
try: client.delete_collection('products_sbert')
except: pass
sbert_collection = client.create_collection('products_sbert', metadata={'hnsw:space': 'cosine'})

# Add in batches
BATCH = 100
for i in range(0, len(df_sample), BATCH):
    sl = df_sample.iloc[i:i+BATCH]
    ids = [str(r['id']) for _, r in sl.iterrows()]
    docs = sl['product_text'].tolist()
    metas = [{'name': r['productDisplayName'], 'category': r['masterCategory'],
               'subCategory': r['subCategory'], 'articleType': r['articleType'],
               'colour': r['baseColour']} for _, r in sl.iterrows()]
    clip_collection.add(embeddings=clip_embeddings[i:i+BATCH].tolist(), documents=docs, metadatas=metas, ids=ids)
    sbert_collection.add(embeddings=sbert_embeddings[i:i+BATCH].tolist(), documents=docs, metadatas=metas, ids=ids)

print(f'✅ ChromaDB loaded: {clip_collection.count()} products indexed')

---
# 🎯 TASK 1: Smart Product Recommendation Engine
> Finds complementary products using CLIP embeddings + category-aware cross-selling logic

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics.pairwise import cosine_similarity

# Complementary category map — the "bought together" logic
COMPLEMENTARY_MAP = {
    'Footwear':    ['Socks', 'Sportswear', 'Accessories', 'Bottomwear'],
    'Topwear':     ['Bottomwear', 'Accessories', 'Footwear', 'Belts'],
    'Bottomwear':  ['Topwear', 'Footwear', 'Belts', 'Accessories'],
    'Accessories': ['Topwear', 'Watches', 'Footwear'],
    'Watches':     ['Accessories', 'Topwear', 'Footwear'],
    'Sportswear':  ['Footwear', 'Accessories', 'Socks'],
    'Bags':        ['Accessories', 'Topwear', 'Footwear'],
    'Innerwear':   ['Topwear', 'Bottomwear'],
}

def recommend_complementary(product_name, top_k=5):
    """
    Given a product name, find the most complementary products.
    Strategy:
      1. Find the product's embedding via CLIP
      2. Identify its category
      3. Pull candidates from complementary categories
      4. Rank by semantic similarity (cross-category affinity)
    """
    # Encode the query
    q_emb = get_clip_text_embeddings([product_name])

    # Find the product's own category from the dataset
    match = df_sample[df_sample['productDisplayName'].str.contains(product_name, case=False, na=False)]
    if len(match) == 0:
        own_cat = 'Footwear'  # default
    else:
        own_cat = match.iloc[0]['subCategory']

    comp_cats = COMPLEMENTARY_MAP.get(own_cat, list(COMPLEMENTARY_MAP.keys()))

    # Filter to complementary categories
    candidates = df_sample[df_sample['subCategory'].isin(comp_cats)].copy()
    if len(candidates) == 0:
        candidates = df_sample.copy()

    cand_embs = clip_embeddings[candidates.index]
    sims = cosine_similarity(q_emb, cand_embs)[0]
    candidates = candidates.copy()
    candidates['similarity'] = sims
    top = candidates.nlargest(top_k, 'similarity')

    return top[['productDisplayName', 'masterCategory', 'subCategory', 'baseColour', 'similarity']]


def visualize_recommendations(product_name, recs):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Complementary Recommendations for: "{product_name}"', fontsize=14, fontweight='bold')

    # Bar chart — similarity scores
    ax = axes[0]
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(recs))[::-1])
    bars = ax.barh(recs['productDisplayName'].str[:30], recs['similarity'], color=colors, edgecolor='white')
    ax.set_xlabel('Semantic Affinity Score', fontsize=11)
    ax.set_title('Ranked Complementary Products', fontsize=12)
    ax.set_xlim(0, 1)
    for bar, val in zip(bars, recs['similarity']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.invert_yaxis()
    ax.spines[['top','right']].set_visible(False)

    # Category breakdown pie
    ax2 = axes[1]
    cat_counts = recs['subCategory'].value_counts()
    wedge_colors = plt.cm.Set3(np.linspace(0, 1, len(cat_counts)))
    ax2.pie(cat_counts.values, labels=cat_counts.index, autopct='%1.0f%%',
            colors=wedge_colors, startangle=140, textprops={'fontsize': 10})
    ax2.set_title('Recommendation Category Mix', fontsize=12)

    plt.tight_layout()
    plt.savefig('task1_recommendations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: task1_recommendations.png')


# --- RUN ---
query_product = 'Running Shoe'
recs = recommend_complementary(query_product, top_k=5)
print(f'\n🎯 Complementary recommendations for: "{query_product}"\n')
print(recs.to_string(index=False))
visualize_recommendations(query_product, recs)

---
# 🔍 TASK 2: Unique Product Catalog Creation
> Deduplicates near-identical products using SBERT embeddings + DBSCAN clustering

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_distances
import seaborn as sns

def deduplicate_catalog(product_list, similarity_threshold=0.15):
    """
    Given a raw product list, identify and remove duplicates.
    Strategy:
      1. Embed all products with SBERT
      2. Compute pairwise cosine distances
      3. DBSCAN clusters near-duplicates automatically
      4. Pick canonical representative per cluster
    """
    embs = normalize(sbert_model.encode(product_list))
    dist_matrix = cosine_distances(embs)

    db = DBSCAN(eps=similarity_threshold, min_samples=1, metric='precomputed')
    labels = db.fit_predict(dist_matrix)

    clusters = {}
    for idx, label in enumerate(labels):
        clusters.setdefault(label, []).append(idx)

    unique_catalog = []
    cluster_info = []
    for label, indices in clusters.items():
        # Pick shortest/cleanest name as canonical
        canonical_idx = min(indices, key=lambda i: len(product_list[i]))
        canonical = product_list[canonical_idx]
        members = [product_list[i] for i in indices]

        # Average intra-cluster similarity
        if len(indices) > 1:
            sub = embs[indices]
            sim = cosine_similarity(sub).mean()
        else:
            sim = 1.0

        unique_catalog.append(canonical)
        cluster_info.append({'canonical': canonical, 'members': members, 'avg_similarity': sim, 'size': len(indices)})

    return unique_catalog, cluster_info, embs, labels


def visualize_dedup(product_list, labels, embs, cluster_info):
    from sklearn.decomposition import PCA

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Task 2: Product Deduplication Results', fontsize=14, fontweight='bold')

    # 2D PCA projection of embeddings, colored by cluster
    ax = axes[0]
    pca = PCA(n_components=2)
    coords = pca.fit_transform(embs)
    unique_labels = list(set(labels))
    palette = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    for i, lbl in enumerate(unique_labels):
        mask = labels == lbl
        ax.scatter(coords[mask, 0], coords[mask, 1], c=[palette[i]], s=120,
                   label=f'Cluster {lbl}', edgecolors='white', linewidths=0.5)
    for j, name in enumerate(product_list):
        ax.annotate(name[:20], (coords[j, 0], coords[j, 1]), fontsize=7,
                    xytext=(4, 4), textcoords='offset points', alpha=0.8)
    ax.set_title('Semantic Clusters (PCA 2D)', fontsize=12)
    ax.set_xlabel('PCA 1'); ax.set_ylabel('PCA 2')
    ax.spines[['top','right']].set_visible(False)

    # Cluster size vs similarity
    ax2 = axes[1]
    sizes = [c['size'] for c in cluster_info]
    sims = [c['avg_similarity'] for c in cluster_info]
    names = [c['canonical'][:20] for c in cluster_info]
    sc = ax2.scatter(sizes, sims, c=sims, cmap='RdYlGn', s=200, edgecolors='gray', linewidths=0.5, vmin=0.5, vmax=1.0)
    for i, name in enumerate(names):
        ax2.annotate(name, (sizes[i], sims[i]), fontsize=8, xytext=(5, 3), textcoords='offset points')
    plt.colorbar(sc, ax=ax2, label='Avg Intra-cluster Similarity')
    ax2.set_xlabel('Cluster Size (# duplicates)', fontsize=11)
    ax2.set_ylabel('Avg Similarity Score', fontsize=11)
    ax2.set_title('Cluster Quality Overview', fontsize=12)
    ax2.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('task2_deduplication.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: task2_deduplication.png')


# --- RUN ---
raw_products = [
    'Blue Shirt A', 'Blue Shirt B', 'Blue Shirt C',
    'Running Shoe A', 'Running Shoe B', 'Athletic Running Sneaker',
    'Slim Fit Denim Jeans', 'Dark Wash Jeans', 'Denim Trousers',
    'Sports Watch Silver', 'Digital Watch A', 'Smartwatch Pro',
    'Black Hoodie', 'Dark Hooded Sweatshirt'
]

catalog, cluster_info, embs, labels = deduplicate_catalog(raw_products)

print(f'Input: {len(raw_products)} products')
print(f'After dedup: {len(catalog)} unique products ({100 - int(len(catalog)/len(raw_products)*100)}% reduction)\n')

print('=== Clusters Found ===')
for c in cluster_info:
    print(f'  [Cluster] Canonical: "{c["canonical"]}" | Members: {c["members"]} | Avg Sim: {c["avg_similarity"]:.3f}')

print(f'\n=== Final Clean Catalog ===')
for i, p in enumerate(catalog, 1):
    print(f'  {i}. {p}')

visualize_dedup(raw_products, labels, embs, cluster_info)

---
# 🔎 TASK 3: Reverse Product Search (Text → Products)
> CLIP cross-modal text-to-product semantic search

In [ ]:
def text_search(query, top_k=6, use_clip=True):
    """
    Search products using a natural language text query.
    Uses CLIP text embeddings to match against indexed product embeddings.
    CLIP aligns text and image in the same embedding space — this is the
    same mechanism used in zero-shot image classification.
    """
    if use_clip:
        q_emb = get_clip_text_embeddings([query])
        sims = cosine_similarity(q_emb, clip_embeddings)[0]
    else:
        q_emb = normalize(sbert_model.encode([query]))
        sims = cosine_similarity(q_emb, sbert_embeddings)[0]

    results = df_sample.copy()
    results['similarity'] = sims
    top = results.nlargest(top_k, 'similarity')
    return top[['productDisplayName', 'masterCategory', 'subCategory', 'baseColour', 'similarity']]


def visualize_search(query, results):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Text Search Results for: "{query}"', fontsize=14, fontweight='bold')

    # Ranked results bar chart
    ax = axes[0]
    colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(results))]
    bars = ax.barh(results['productDisplayName'].str[:35], results['similarity'],
                   color=colors, edgecolor='white')
    ax.set_xlabel('CLIP Semantic Similarity', fontsize=11)
    ax.set_title('Top Matching Products', fontsize=12)
    ax.set_xlim(0, 1)
    for bar, val in zip(bars, results['similarity']):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.invert_yaxis()
    ax.spines[['top','right']].set_visible(False)

    # Colour distribution of results
    ax2 = axes[1]
    colour_counts = results['baseColour'].value_counts()
    colour_map = {'Blue': '#2196F3', 'Black': '#333', 'White': '#EEEEEE',
                  'Red': '#F44336', 'Green': '#4CAF50', 'Grey': '#9E9E9E',
                  'Navy Blue': '#1A237E', 'Brown': '#795548'}
    bar_colors = [colour_map.get(c, '#90A4AE') for c in colour_counts.index]
    ax2.bar(colour_counts.index, colour_counts.values, color=bar_colors, edgecolor='white')
    ax2.set_xlabel('Product Colour', fontsize=11)
    ax2.set_ylabel('Count', fontsize=11)
    ax2.set_title('Colour Distribution in Results', fontsize=12)
    ax2.spines[['top','right']].set_visible(False)
    plt.xticks(rotation=30, ha='right')

    plt.tight_layout()
    plt.savefig('task3_text_search.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: task3_text_search.png')


# --- RUN multiple queries ---
queries = ['blue casual shirt', 'sports shoes for running', 'warm winter jacket']

for q in queries:
    results = text_search(q, top_k=6)
    print(f'\n🔎 Query: "{q}"')
    print(results.to_string(index=False))
    visualize_search(q, results)

---
## 📊 Summary: Architecture Overview

| Task | Model | Technique | Output |
|------|-------|-----------|--------|
| Task 1: Recommendation | CLIP | Cross-category cosine similarity | 5 complementary products |
| Task 2: Deduplication | SBERT | DBSCAN clustering on embedding space | Clean unique catalog |
| Task 3: Text Search | CLIP | Cross-modal text↔product similarity | Ranked search results |

**Why CLIP?** CLIP aligns text and images in a shared embedding space — the same text description of a product lands near its visual representation. This enables zero-shot, keyword-free semantic search.

**Why SBERT for dedup?** Sentence-BERT produces more fine-grained sentence similarity than CLIP, better capturing subtle name variations like 'Running Shoe A' vs 'Athletic Running Sneaker'.

**Why ChromaDB?** It gives us persistent, scalable vector storage with cosine similarity search — production-ready for millions of products.